## TL;DR

This notebook orchestrates the cleaning + feature engineering pipeline that turns the raw scraped job details into an analysis-ready master dataset.

**Input:** `data/raw/job_details.csv` (raw scraped job postings from :mod:`glints_scraper.details`)

**Output:** `data/processed/master_dataset_eda.csv` (cleaned + feature-engineered)

**Where the logic lives:** :mod:`glints_scraper.transform`. This notebook is intentionally thin — it just stages the inputs and surfaces the pipeline result. The actual parsing, skill detection, and field normalisation are unit-tested in `tests/test_transform.py`.

**Key transformations:**
- Salary string -> numeric (`salary_min`, `salary_max`, `salary_avg`)
- Skill tags + description -> 32 boolean skill columns (`skill_<name>`)
- Location string -> canonical `city_clean`
- Experience string -> `exp_min_years` (Int64)
- Education / employment / workplace -> canonical categorical columns

**Why this matters:** Downstream EDA and the Streamlit dashboard consume this dataset directly. Clean column names + numeric types are prerequisites for any visualisation.

# Cleaning + Feature Engineering

This stage cleans and transforms the scraped job-detail data so it is ready for Exploratory Data Analysis (EDA). The workflow is:

1. Drop rows missing required fields (skill tags, education, method, experience).
2. Parse `job_salary` (IDR) into `salary_min`, `salary_max`, `salary_avg`.
3. Drop closed listings and parse `exp_min_years`.
4. Detect skills across `job_description + skill_tags` -> 32 boolean flags.
5. Normalise `job_education` -> `job_education_clean`.
6. Classify `employment_type` from `job_method`.
7. Classify `workplace_type` from `job_title + job_description`.

Every step is implemented as a pure function in :mod:`glints_scraper.transform` and is unit-tested. This notebook runs the pipeline end-to-end and saves the result for downstream consumers.

In [ ]:
import sys
from pathlib import Path

# Allow the notebook to import the package from `src/` without an editable install.
sys.path.insert(0, str(Path.cwd() / "src"))

import pandas as pd

from glints_scraper.transform import transform

## 1. Load Raw Data

Read the job-detail file produced by the Phase 2 detail scraper and inspect the structure before applying any transformations.

In [ ]:
raw_path = Path("data/raw/job_details.csv")
assert raw_path.exists(), f"Run `make scrape-details` first; missing {raw_path}"
raw = pd.read_csv(raw_path)
print(f"raw shape: {raw.shape}")
raw.head()

## 2. Run the Cleaning Pipeline

A single call to :func:`transform` runs every stage. The function is pure — it does not mutate the input frame and returns the canonical schema.

In [ ]:
master = transform(raw)
print(f"master shape: {master.shape}")
print(f"columns: {list(master.columns)[:15]}...")
master.head()

## 3. Validate the Result

Spot-check the canonical schema and key derived columns so we catch any upstream regressions before the EDA stage reads the file.

In [ ]:
expected_core = {
    "keyword", "job_title", "company_name", "city_clean",
    "salary_min", "salary_max", "salary_avg",
    "exp_min_years", "employment_type", "workplace_type",
    "job_education_clean",
}
skill_cols = [c for c in master.columns if c.startswith("skill_")]

print(f"rows: {len(master):,}")
print(f"core columns present: {expected_core.issubset(master.columns)}")
print(f"skill flags: {len(skill_cols)}")
print(f"workplace_type distribution:\n{master['workplace_type'].value_counts().to_string()}")
print(f"\neducation distribution:\n{master['job_education_clean'].value_counts().to_string()}")

## 4. Persist the Master Dataset

Write the cleaned frame to `data/processed/master_dataset_eda.csv`. This file is the single source of truth for `eda.ipynb` and the Streamlit dashboard.

In [ ]:
out_path = Path("data/processed/master_dataset_eda.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
master.to_csv(out_path, index=False)
print(f"saved {len(master):,} rows to {out_path}")